# RQ2 — fresh seed-7 Pure-SW replication

One independent Uniform-fixed trajectory produces checkpoints 10/50/100. The checkpoints are then frozen and used only to extract SW and gradient Gram structure. The gate compares Uniform Pair, raw Resource Pair, Pure-SW Pair, and the gradient oracle under identical marginals `pi_i = 1/7`. Accuracy and learned hybrid coefficients are not used.

## Required Kaggle inputs

1. A CIFAR-100 dataset containing `cifar-100-python/train`, `test`, and `meta`.
2. The frozen development-gates output containing `gate_a_summary.json`. The previous seed-6 Gate-B1 ZIP/dataset is also acceptable because its export contains `frozen_development_gates/gate_a_summary.json`.
3. Optional only for resume: a previous partial/full `fresh_seed_7` output.

Seed-6 checkpoints and the frozen learned predictor are not required. Add Kaggle secret `github_token` for the private repository.

In [ ]:
import os, subprocess, sys, json, time, zipfile, importlib, hashlib
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() >= 1, 'Select at least one T4 GPU'
GPU_IDS = tuple(range(min(2, torch.cuda.device_count())))
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('GPUs:', [torch.cuda.get_device_name(i) for i in GPU_IDS])

## Resolve and audit inputs before training

In [ ]:
import rq2_fresh_puresw_replication as replication
replication = importlib.reload(replication)
INPUT_ROOT = Path('/kaggle/input')
DATASET_ROOT = replication.find_cifar100_root(INPUT_ROOT)
GATE_A_SUMMARY = replication.find_gate_a_summary(INPUT_ROOT)
print('CIFAR-100 root:', DATASET_ROOT)
print('Frozen Gate-A FLOPs:', GATE_A_SUMMARY)
gate_a = json.loads(GATE_A_SUMMARY.read_text())
assert len(gate_a['flops']) == 14
print('Input audit PASS. No seed-6 checkpoint or frozen learned predictor will be used.')

## Materialize optional seed-7 progress and freeze protocol

The run is exactly resumable. With no seed-7 input it starts from scratch; an attached matching seed-7 output resumes optimizer/scheduler/RNG rather than restarting.

In [ ]:
FRESH_ROOT = replication.materialize_progress(
    INPUT_ROOT, '/kaggle/working/fresh_seed_7', '/kaggle/working/materialized-fresh-seed7'
)
CONFIG = replication.load_config(
    PROJECT_ROOT/'configs/kaggle_s1_extension_100.yaml', DATASET_ROOT
)
locked = {
 'status':'FROZEN_BEFORE_FRESH_TRAINING', 'seed':7,
 'trajectory':'fresh_uniform_fixed_puresw_replication',
 'anchors':[0.25,0.50,0.75,1.00], 'interior_widths':[round(0.30+0.05*i,2) for i in range(14)],
 'schedule':'50+50_optimizer_scheduler_reset', 'checkpoints':[10,50,100],
 'kd_lambda':1.0, 'kd_temperature':2.0, 'batch_size':128,
 'num_sw_projections':128, 'num_gradient_batches':8,
 'pair_marginal':1/7, 'accuracy_used_for_gate':False,
 'learned_predictor_used':False, 'gate_c_authorized':False,
 'gate_a_sha256':hashlib.sha256(GATE_A_SUMMARY.read_bytes()).hexdigest(),
 'git_commit':GIT_COMMIT
}
FRESH_ROOT.mkdir(parents=True, exist_ok=True)
protocol_path = FRESH_ROOT/'frozen_fresh_protocol.json'
if protocol_path.exists():
    previous = json.loads(protocol_path.read_text())
    keys = [key for key in locked if key != 'git_commit']
    assert all(previous.get(key) == locked[key] for key in keys), 'Attached seed-7 progress violates frozen protocol'
else:
    protocol_path.write_text(json.dumps(locked, indent=2)+'\n')
print(json.dumps(locked, indent=2))

## Phase 1 — train one independent trajectory to epoch 100

Training uses GPU 0. Epochs 1–50 use LR 0.1; epoch 51 resets optimizer/scheduler and uses LR 0.01. Snapshots are immutable at 10/50/100.

In [ ]:
started = time.perf_counter()
replication.train_trajectory(CONFIG, FRESH_ROOT)
print(f'Training completed/resumed in {(time.perf_counter()-started)/3600:.2f} hours')
for epoch in (10,50,100):
    path = FRESH_ROOT/'checkpoints'/f'epoch_{epoch:03d}.pt'
    assert path.is_file(); print(path, f'{path.stat().st_size/2**20:.1f} MiB')

## Phases 2–4 — frozen-state SW and gradient extraction

With T4×2, F10 and F50 start in parallel and F100 uses the first released GPU. With one T4 the same jobs run sequentially. No optimizer step is performed.

In [ ]:
import scripts.run_fresh_puresw_replication as runner
runner = importlib.reload(runner)
workers = runner.run_workers(
    FRESH_ROOT, DATASET_ROOT, GATE_A_SUMMARY, gpu_ids=GPU_IDS
)
print('Completed workers:', workers)

## Phases 5–6 — fixed-marginal LPs and exact variance gate

This stage is CPU-only. PASS requires `V_SW < V_uniform` at least 2/3 checkpoints.

In [ ]:
summary = replication.merge_and_evaluate(FRESH_ROOT, workers)
print(json.dumps(summary, indent=2))
import pandas as pd
from IPython.display import display, Image, Markdown
display(pd.read_csv(FRESH_ROOT/'evaluation/fresh_seed7_puresw_replication.csv'))
display(pd.read_csv(FRESH_ROOT/'evaluation/fresh_seed7_bootstrap_contrasts.csv'))
display(Image(filename=str(FRESH_ROOT/'evaluation/fresh_seed7_puresw_variance.png')))
display(Markdown((FRESH_ROOT/'evaluation/fresh_seed7_puresw_replication_summary.md').read_text()))
assert summary['predictor_fitted_or_used'] is False
assert summary['gate_c_end_to_end_authorized'] is False

## Validate and export resumable result

In [ ]:
required = [
 'checkpoints/epoch_010.pt','checkpoints/epoch_050.pt','checkpoints/epoch_100.pt',
 'pair_structure.csv','sw_matrices/epoch_010.npy','sw_matrices/epoch_050.npy','sw_matrices/epoch_100.npy',
 'gradient_grams/epoch_010.npy','gradient_grams/epoch_050.npy','gradient_grams/epoch_100.npy',
 'evaluation/fresh_seed7_puresw_replication.csv',
 'evaluation/fresh_seed7_pair_policies.csv',
 'evaluation/fresh_seed7_bootstrap_contrasts.csv',
 'evaluation/fresh_seed7_puresw_replication_summary.json',
 'evaluation/fresh_seed7_puresw_replication_summary.md',
 'evaluation/fresh_seed7_puresw_variance.png',
]
missing = [name for name in required if not (FRESH_ROOT/name).is_file() or (FRESH_ROOT/name).stat().st_size == 0]
assert not missing, f'Missing outputs: {missing}'
bundle = Path('/kaggle/working/rq2-fresh-seed7-puresw-replication.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for path in FRESH_ROOT.rglob('*'):
        if path.is_file(): archive.write(path, Path('fresh_seed_7')/path.relative_to(FRESH_ROOT))
    archive.write(GATE_A_SUMMARY, Path('frozen_inputs')/'gate_a_summary.json')
print('Download/persist:', bundle, f'{bundle.stat().st_size/2**30:.2f} GiB')
bundle